<a href="https://colab.research.google.com/github/Vidhya-95/news-classification-transformers-rag/blob/main/AI_Powered_News_Insight_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

vidhyashinde_bbc_news_dataset_path = kagglehub.dataset_download('vidhyashinde/bbc-news-dataset')

print('Data source import complete.')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score,ConfusionMatrixDisplay
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer


# Data Understanding & Preprocessing

## Loading Dataset

In [ ]:
bbc_news_og_ds = pd.read_csv("/kaggle/input/datasets/vidhyashinde/bbc-news-dataset/BBC_Capstone_Dataset.csv")

In [ ]:
bbc_news = bbc_news_og_ds.copy()

In [ ]:
bbc_news.head()

In [ ]:
bbc_news.shape

In [ ]:
bbc_news.columns

In [ ]:
bbc_news.info()

## Visualizing the category column

In [ ]:
bbc_news['category'].value_counts()

In [ ]:
category = bbc_news['category'].value_counts()
category_name = category.index.to_list()
category_counts = category.values.tolist()
plt.figure(figsize=(15,8))
plt.bar(category_name,category_counts)
plt.xlabel('Caterory')
plt.xticks(rotation=90)
plt.ylabel('Counts')
plt.title("News Distribution")
plt.show()

## Text Cleaning
In this step we will convert text to lowercase and removing punctuations,numbers and stopwords.
Since we are performing text classification, removing stopwords help us focus on relevant words which provide semantic value to text analysis.

In [ ]:
# def remove_punctuation_numbers(text):
#     tokens = word_tokenize(text)
#     clean_text = ' '.join([word for word in tokens if word.isalpha()])
#     return clean_text
#     #return text.translate(str.maketrans('', '', string.punctuation))

# bbc_news['text_without_punctuations']=bbc_news['text'].apply(remove_punctuation_numbers)

# len(bbc_news['text_without_punctuations'][0].split(' '))

# stop_words = set(stopwords.words('english'))
# def remove_stopwords(text):
#     tokens = word_tokenize(text)
#     clean_text = ' '.join([word for word in tokens if word not in stop_words])
#     return clean_text

# bbc_news['text_without_stopwords']=bbc_news['text_without_punctuations'].apply(remove_stopwords)

# len(bbc_news['text_without_stopwords'][0].split(' '))

In [ ]:
# Removing punctuations,numbers and stopwords
bbc_news['text']=bbc_news['text'].str.lower()
stop_words = set(stopwords.words('english'))
def clean_text(text):
    tokens = word_tokenize(text)
    clean_text = ' '.join([word for word in tokens if word.isalpha() and word not in stop_words])
    return clean_text

In [ ]:
bbc_news['processed_text']=bbc_news['text'].apply(clean_text)
bbc_news = bbc_news.drop('text',axis=1)

## Label Encoding
Converting news categories into numerical labels.
Categories are mapped as following:

* 'business': 0,
* 'entertainment': 1,
* 'politics': 2,
* 'sport': 3,
* 'tech': 4


In [ ]:
le = LabelEncoder()
bbc_news['encoded_category'] = le.fit_transform(bbc_news['category'])

bbc_news = bbc_news.drop('category',axis=1)

In [ ]:
# le_name_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
# print(le_name_mapping)

# Applying Machine Learning

In [ ]:
#Splitting dataset into training(90%) and testing(10%)
X_train,X_test,y_train,y_test = train_test_split(bbc_news['processed_text'],bbc_news['encoded_category'],random_state=42,test_size=0.1)

## TF-IDF Vectorization
Converting our raw text data to numerical vectors.


* fit_transform() on X_train: Learns the vocabulary and calculates IDF (Inverse Document Frequency) scores exclusively from the training set.
* transform() on X_test: Converts test data into vectors based only on the training vocabulary. If you fit the vectorizer on the test set, your model gets indirect clues about test data, resulting in artificially high accuracy metrics that will fail in the real world.


In [ ]:
tf_idf_vec = TfidfVectorizer()

X_train_tfidf = tf_idf_vec.fit_transform(X_train)
X_test_tfidf = tf_idf_vec.transform(X_test)

## Training ML model - Logistic Regression

In [ ]:
logistic_reg = LogisticRegression()

In [ ]:
logistic_reg.fit(X_train_tfidf,y_train)

In [ ]:
y_pred = logistic_reg.predict(X_test_tfidf)

## Model Evaluation
Evaluating accuracy score, confusion matrix,classification report

In [ ]:
print("Accuracy_score" ,accuracy_score(y_test, y_pred))

In [ ]:
print("Classification Report \n",classification_report(y_test,y_pred))

In [ ]:
cm = confusion_matrix(y_test,y_pred)
print("Confusion Matrix \n",cm)

In [ ]:
logistic_reg.classes_

In [ ]:
# Creating a visually appealing confusion matrix
#Category mappings ==> 'business': 0,'entertainment': 1,'politics': 2,'sport': 3,'tech': 4
plt.figure(figsize=(8, 8))
cm_display = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=logistic_reg.classes_)
cm_display.plot()
plt.show()